# 11 · 설비 이상 탐지와 고장 예측

원본 실습의 Z-score, Isolation Forest, 분류 모델, RUL 개념을 한 흐름으로 비교합니다. 이 노트북은 `scikit-learn`이 필요합니다. 예제는 계산 연습용이며 실제 설비의 안전 기준이 아닙니다.

> 위에서 아래로 실행하세요. 예제 데이터는 노트북 안에서 만듭니다. 코드 셀 아래의 출력으로 결과를 확인하고, 실제 데이터에서는 열 이름·단위·기간을 먼저 확인하세요.

## 정상 기준과 Z-score

**코드 → 코드 개념**: 정상 구간에서 평균·표준편차를 학습하고 이후 값과 비교한다.

**코드 사용법**: 정상 기준을 고정해 새 측정값의 점수를 계산한다.

In [ ]:
import numpy as np
import pandas as pd
normal = pd.Series([2.0, 2.1, 2.2, 2.3, 2.4])
future = pd.Series([2.3, 2.7, 3.5])
center, scale = normal.mean(), normal.std(ddof=0)
z = (future - center) / scale
print(center, scale, z.round(2))

**같은 결과를 얻는 방법과 선택 이유**

- 정상 기간만 기준에 넣어야 미래 이상이 평균·표준편차를 왜곡하지 않는다.
- Z-score는 단일 센서의 단순 기준에 좋다. 여러 센서가 함께 변하는 패턴은 다변량 모델을 고려한다. 기준값이 0이면 계산할 수 없다.

## Isolation Forest

**코드 → 코드 개념**: Isolation Forest는 라벨 없이 다변량 관측에서 상대적으로 고립된 점을 찾는다.

**코드 사용법**: 정상에 가까운 학습 표본으로 모델을 맞추고 새 값에 점수를 낸다.

In [ ]:
from sklearn.ensemble import IsolationForest
train = pd.DataFrame({"rms": [2.0, 2.1, 2.2, 2.3, 2.4, 2.2, 2.1],
                      "temperature": [70, 71, 72, 73, 74, 71, 72]})
new = pd.DataFrame({"rms": [2.2, 5.0], "temperature": [72, 95]})
model = IsolationForest(contamination=0.1, random_state=42)
model.fit(train)
print(model.predict(new))  # 1: 정상 측, -1: 이상 측
print(model.decision_function(new))

**같은 결과를 얻는 방법과 선택 이유**

- `predict`는 간단한 라벨, `decision_function`은 상대적 점수다. 점수와 원본 센서값을 같이 봐야 해석 가능하다.
- `contamination`은 예상 이상 비율과 연결되므로 경보량을 검증한다. 학습 데이터에 고장이 많이 섞이면 기준이 흐려진다.

## 분할과 정보 누출

**코드 → 코드 개념**: 학습 데이터와 평가 데이터는 시간·설비 단위로 분리해야 실제 미래 예측에 가깝다.

**코드 사용법**: 같은 설비의 미래 측정을 평가 집합으로 둔다.

In [ ]:
sequence = pd.DataFrame({"cycle": range(1, 11), "sensor": [2.0, 2.1, 2.0, 2.2, 2.3, 2.5, 2.8, 3.0, 3.3, 3.7]})
train_part = sequence.loc[sequence["cycle"] <= 6]
test_part = sequence.loc[sequence["cycle"] > 6]
print(train_part["cycle"].tolist(), test_part["cycle"].tolist())

**같은 결과를 얻는 방법과 선택 이유**

- 무작위 `train_test_split`은 독립된 행이라면 편하지만 같은 설비의 인접 시점이 양쪽에 섞이면 평가가 과하게 좋아질 수 있다.
- 다른 설비로 일반화할 목표라면 설비 ID별 분리, 같은 설비의 미래를 예측할 목표라면 시간순 분리를 쓴다. 스케일러와 결측 대체도 학습 집합에서만 `fit`한다.

## 지도학습 분류

**코드 → 코드 개념**: 고장 라벨이 있으면 `fit(X, y)`로 입력과 정답의 관계를 학습한다.

**코드 사용법**: 작은 예제에서 확률과 분류 결과를 확인한다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
X_train = pd.DataFrame({"rms": [1.8, 2.0, 2.1, 2.3, 3.8, 4.0, 4.2, 4.5],
                        "temperature": [68, 70, 72, 74, 85, 88, 90, 92]})
y_train = pd.Series([0, 0, 0, 0, 1, 1, 1, 1])
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)
X_new = pd.DataFrame({"rms": [2.2, 4.1], "temperature": [73, 89]})
print(clf.predict(X_new), clf.predict_proba(X_new))

**같은 결과를 얻는 방법과 선택 이유**

- `predict`는 클래스, `predict_proba`는 클래스별 추정 확률이다. 경보 비용에 맞춰 확률 기준을 조정할 수 있다.
- 지도학습은 신뢰할 만한 고장 라벨이 있을 때 유리하다. 라벨이 부족하면 이상 탐지나 규칙 기반 점검을 먼저 고려한다.

## RUL과 평가

**코드 → 코드 개념**: RUL은 남은 유효 수명이다. 현재 cycle에서 실제 고장 cycle까지의 차이로 만들 수 있다.

**코드 사용법**: 간단한 RUL 라벨과 혼동행렬을 확인한다.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
life = pd.DataFrame({"unit": [1, 1, 1, 2, 2], "cycle": [1, 2, 3, 1, 2]})
life["failure_cycle"] = life.groupby("unit")["cycle"].transform("max")
life["RUL"] = life["failure_cycle"] - life["cycle"]
print(life)
actual = [0, 0, 1, 1]
predicted = [0, 1, 1, 0]
print(confusion_matrix(actual, predicted))
print(classification_report(actual, predicted, zero_division=0))

**같은 결과를 얻는 방법과 선택 이유**

- 실제 run-to-failure 데이터에서만 그룹 최대 cycle을 고장 시점으로 볼 수 있다. 중도 종료된 설비의 마지막 관측 시점은 고장 시점이 아니다.
- 혼동행렬은 놓친 고장(FN)과 오경보(FP)를 분리한다. RUL 예측 자체는 회귀 문제이므로 MAE 같은 수명 오차도 평가한다.

## 원본 학습 자료

[`2. practice/06_Z-score`](../2.%20practice/06_Z-score), [`2. practice/07_domain_Predictive_Maintenance`](../2.%20practice/07_domain_Predictive_Maintenance)